### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [24]:
import numpy as np
import pandas as pd

### Punto 1

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

Elegimos 5 documentos aleatorios del dataset de entrenamiento. Para garantizar que los documentos sean legibles se seleccionaron a mano los 5 documentos. Sin embargo; se deja documentado como obtener los documentos aleatoriamente usando [`numpy.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.rand.html)

In [25]:
# Elijo 5 documentos a mano que sean faciles de ver
rand_indexes = [1, 42, 51, 90, 291]

# Usando np.random podemos simular 5 documentos aleatorios
# np.random.seed(42)
# rand_indexes = np.random.randint(low = 0, high = 1314, size = 5)

sample = [ newsgroups_train['data'][idx] for idx in rand_indexes ]

display(pd.DataFrame(sample, columns=["doc"]))

,doc
0,A fair number of brave souls who upgraded thei...
1,Western Digital 1-800-832-4778.....Sam
2,I have been following this thread on talk.reli...
3,Does anyone know what processor the Atari 2600...
4,"Sorry to everyone for wasting space. Matt, th..."


Para cada documento medimos la similitud con respecto a los demas documentos usando la similitud coseno.

In [26]:

def get_top_5_similar_docs(sim_docs: np.ndarray, indx2doc: list[str], index2target: list[str], y_train: np.ndarray):
    top_5_similar_index = np.argsort(sim_docs)[::-1][1:6]
    return { 
        "similar_docs" : [indx2doc[idx] for idx in top_5_similar_index],
        "simil_target" : [index2target[y_train[idx]] for idx in top_5_similar_index]
    }

# Usamos el dataset vectorizado usando tfidfvect
# X_train = tfidfvect.fit_transform(newsgroups_train.data)
cos_sim_sample = [
    {
        "book": newsgroups_train['data'][int(idx)],
        "target": newsgroups_train.target_names[y_train[idx]],
        **get_top_5_similar_docs(
            sim_docs = cosine_similarity(X_train[idx], X_train)[0],
            indx2doc = newsgroups_train['data'],
            index2target = newsgroups_train['target_names'],
            y_train = y_train
        )
    } 
    for idx in rand_indexes 
]

df = pd.DataFrame(cos_sim_sample).explode(["similar_docs", "simil_target"], ignore_index=False)
df

,book,target,similar_docs,simil_target
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I am continuing to collect user results to pro...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I have read one report of a brave soul who rew...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,After reading reports from Germany of success ...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I've just completed a successful upgrade of a ...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,dhk@ubbpc.uucp (Dave Kitabjian) writes ...\n\n...,comp.sys.mac.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,1-800-832-4778 Western Digital's Voice Mail -\...,comp.sys.ibm.pc.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,Here's a list of 800 numbers I have compiled f...,comp.sys.mac.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"\n\n You're correct, except that's Quadra 8...",comp.sys.mac.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"Western Digital 3.5"" IDE 40 Meg Hard drive.\n$...",misc.forsale
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"\ni'm telling you, sam, three l's. call up mo...",rec.sport.baseball


Observamos que para el documento 1 todos los documentos similares pertenecen a la misma clase que el target

In [27]:
df.loc[0]

,book,target,similar_docs,simil_target
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I am continuing to collect user results to pro...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I have read one report of a brave soul who rew...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,After reading reports from Germany of success ...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,I've just completed a successful upgrade of a ...,comp.sys.mac.hardware
0,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware,dhk@ubbpc.uucp (Dave Kitabjian) writes ...\n\n...,comp.sys.mac.hardware


Para el documento 2, solamente el top 3 documentos mas similares pertenecen a la misma clase que el target. Los demas documentos no tienen mucha relacion

In [28]:
df.loc[1]

,book,target,similar_docs,simil_target
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,1-800-832-4778 Western Digital's Voice Mail -\...,comp.sys.ibm.pc.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,Here's a list of 800 numbers I have compiled f...,comp.sys.mac.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"\n\n You're correct, except that's Quadra 8...",comp.sys.mac.hardware
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"Western Digital 3.5"" IDE 40 Meg Hard drive.\n$...",misc.forsale
1,Western Digital 1-800-832-4778.....Sam,comp.sys.ibm.pc.hardware,"\ni'm telling you, sam, three l's. call up mo...",rec.sport.baseball


Para el documento 3, 4 documentos similares pertenecen a la misma clase que el target, pero todos estan en la misma tematica

In [29]:
df.loc[2]

,book,target,similar_docs,simil_target
2,I have been following this thread on talk.reli...,soc.religion.christian,"\n\nFirst of all, ""ceremonial law"" is an extra...",soc.religion.christian
2,I have been following this thread on talk.reli...,soc.religion.christian,Someone sent me this FAQ by E-mail and I post ...,soc.religion.christian
2,I have been following this thread on talk.reli...,soc.religion.christian,[In response to some of the discussions on the...,soc.religion.christian
2,I have been following this thread on talk.reli...,soc.religion.christian,"\n\nJesus also recognized other holy days, lik...",talk.religion.misc
2,I have been following this thread on talk.reli...,soc.religion.christian,\n\nDo you count yourself as one who is weak i...,soc.religion.christian


Para el documento 4, solamente el top 2 documentos mas similares pertenecen a la misma clase que el target. Sin embargo siguen la misma tematica

In [30]:
df.loc[3]

,book,target,similar_docs,simil_target
3,Does anyone know what processor the Atari 2600...,sci.electronics,"\nThe Atari 2600 used a 6502 CPU, just like th...",sci.electronics
3,Does anyone know what processor the Atari 2600...,sci.electronics,For all people that are interested in every as...,sci.electronics
3,Does anyone know what processor the Atari 2600...,sci.electronics,Is it possible to connect a atari monochrome m...,comp.sys.ibm.pc.hardware
3,Does anyone know what processor the Atari 2600...,sci.electronics,\n\ndoes anyone know?\n\n--,sci.med
3,Does anyone know what processor the Atari 2600...,sci.electronics,Have anybody succeded in converting a atari mo...,comp.sys.ibm.pc.hardware


Para el documento 5, ningun documento similar pertenecen a la misma clase que el target

In [31]:
df.loc[4]

,book,target,similar_docs,simil_target
4,"Sorry to everyone for wasting space. Matt, th...",rec.sport.hockey,Please subscribe me to this mailing list,comp.windows.x
4,"Sorry to everyone for wasting space. Matt, th...",rec.sport.hockey,I recently had to move and forgot to update my...,soc.religion.christian
4,"Sorry to everyone for wasting space. Matt, th...",rec.sport.hockey,Would the person who is running the e-mail lis...,rec.sport.baseball
4,"Sorry to everyone for wasting space. Matt, th...",rec.sport.hockey,I'm looking for the address to join the Clevel...,rec.sport.baseball
4,"Sorry to everyone for wasting space. Matt, th...",rec.sport.hockey,Could someone out there please tell me how I c...,rec.autos


### Punto 2

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.


In [32]:
def get_most_similar_doc(sim_docs: np.ndarray, indx2doc: list[str], index2target: list[str], y_train: np.ndarray):
    max_arg = int(np.argmax(sim_docs))
    return { 
        "pred_doc" : indx2doc[max_arg], # Documento con mayor similaridad
        "pred_target": y_train[max_arg],
        "pred_target_label" : index2target[y_train[max_arg]] # Label del doc con mayor similaridad
    }

# Usamos el conjunto previamente vectorizado
# X_train = tfidfvect.fit_transform(newsgroups_train.data)
# X_test = tfidfvect.transform(newsgroups_test.data)
# y_test = newsgroups_test.target
data = [
    {
        "book": newsgroups_test['data'][int(idx)],
        "target": y_test[idx],
        "target_label": newsgroups_test.target_names[y_test[idx]],
        **get_most_similar_doc(
            sim_docs = cosine_similarity(X_test[idx], X_train)[0],
            indx2doc = newsgroups_train['data'],
            index2target = newsgroups_train['target_names'],
            y_train = y_train
        )
    } 
    for idx in range(0, len(newsgroups_test['data'])) 
]

df = pd.DataFrame(data)
display(df)

,book,target,target_label,pred_doc,pred_target,pred_target_label
0,I am a little confused on all of the models of...,7,rec.autos,"The recent rise of nostalgia in this group, co...",0,alt.atheism
1,I'm not familiar at all with the format of the...,5,comp.windows.x,\nIf I have a habit that I really want to brea...,19,talk.religion.misc
2,"\nIn a word, yes.\n",0,alt.atheism,I have one word for you LOSER!!!!,17,talk.politics.mideast
3,\nThey were attacking the Iraqis to drive them...,17,talk.politics.mideast,Accounts of Anti-Armenian Human Right Violatio...,17,talk.politics.mideast
4,\nI've just spent two solid months arguing tha...,19,talk.religion.misc,\n\nI did not claim that our system was object...,0,alt.atheism
...,...,...,...,...,...,...
7527,"\n Henry, if I read you correctly, you may b...",14,sci.space,"A listmember (D Andrew Killie, I think) wrote,...",15,soc.religion.christian
7528,"about\nthem on\n\nActually, I thought Macs wer...",4,comp.sys.mac.hardware,\nExcuse me but... have not all Macs got a CPU!!!,4,comp.sys.mac.hardware
7529,"I sent a version of this post out a while ago,...",9,rec.sport.baseball,Accounts of Anti-Armenian Human Right Violatio...,17,talk.politics.mideast
7530,I have this kit which includes the following :...,6,misc.forsale,"\nNot hard, you can do the refreshing and acce...",12,sci.electronics


Evaluamos el performance del modelo de clasificacion por prototipo y lo contrastamos con el modelo de Naive Bayes previamente entrenado en la Notebook usando la metrica F1-score.

Observamos que tienen un desempeno similar, sin embargo el modelo Naive Bayes tiene una ligera ventaja en ambas metricas

In [33]:
from sklearn.metrics import accuracy_score, f1_score

print("------------\nHeuristica")
print(f"F1-score macro : {f1_score(df['target'], df['pred_target'], average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(df['target'], df['pred_target']):.4f}")
print("------------\nNaive Bayes")
print(f"F1-score macro: {f1_score(y_test, clf.predict(X_test), average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_test, clf.predict(X_test)):.4f}")

------------
Heuristica
F1-score macro : 0.5050
Accuracy: 0.5089
------------
Naive Bayes
F1-score macro: 0.5854
Accuracy: 0.6062


Por ultimo comparamos ambos modelos segun la clase del articulo.

Vemos que la heuristica gana para los articulos 
- `rec.sport.hockey`
- `soc.religion.christian`
- `alt.atheism`
- `talk.politics.misc`
- `talk.religion.misc`

En las 15 clases restantes el modelo Naives Bayes gana

In [34]:
f1_por_clase = pd.DataFrame({
    "clase": newsgroups_test.target_names,
    "soporte": np.bincount(y_test),
    "f1_heuristica": f1_score(df['target'], df['pred_target'], average=None),
    "f1_naive_bayes": f1_score(y_test, clf.predict(X_test), average=None),
}).sort_values("f1_heuristica", ascending=False).reset_index(drop=True)

f1_por_clase

,clase,soporte,f1_heuristica,f1_naive_bayes
0,rec.sport.hockey,399,0.734694,0.713858
1,comp.windows.x,395,0.641975,0.780749
2,rec.sport.baseball,397,0.585970,0.800582
3,sci.crypt,396,0.569682,0.588454
4,rec.motorcycles,398,0.568655,0.734513
5,sci.space,394,0.566145,0.718841
6,sci.med,396,0.561151,0.729651
7,misc.forsale,390,0.532544,0.768137
8,comp.sys.ibm.pc.hardware,392,0.517766,0.637838
9,comp.sys.mac.hardware,385,0.516129,0.693452


### Punto 3
**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

Vamos a probar las siguientes configuraciones de los vectorizadores [CountVectorize](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) y [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [35]:
from itertools import product
configs_vect = list(product(
    ["tfidf", "count"],   # TfidfVectorizer vs CountVectorizer
    [None, "english"],    # stop_words
    [1, 2, 5],            # min_df
    [0.5, 1.0],          # max_df
))
pd.DataFrame(configs_vect, columns=['vectorizer', 'stop_words', 'min_df', 'max_df'])

,vectorizer,stop_words,min_df,max_df
0,tfidf,NaN,1,0.5
1,tfidf,NaN,1,1.0
2,tfidf,NaN,2,0.5
3,tfidf,NaN,2,1.0
4,tfidf,NaN,5,0.5
5,tfidf,NaN,5,1.0
6,tfidf,english,1,0.5
7,tfidf,english,1,1.0
8,tfidf,english,2,0.5
9,tfidf,english,2,1.0


Y vamos a entrenar los modelos [MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB) y [ComplementNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.ComplementNB.html#sklearn.naive_bayes.ComplementNB) modificando su parametro `alpha` en ambos casos.

Compararemos estos modelos usando la metrica de `f1_score` macro

In [36]:
from itertools import product
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# CountVectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
# TfidfVectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
def create_vectorizer(vectorizer_name: str, stop_words: str | None, min_df: float, max_df: float):
    if vectorizer_name == 'tfidf':
        return TfidfVectorizer(stop_words=stop_words, min_df=min_df, max_df=max_df)
    return CountVectorizer(stop_words=stop_words, min_df=min_df, max_df=max_df)

configs_vect = list(product(
    ["tfidf", "count"],   # TfidfVectorizer vs CountVectorizer
    [None, "english"],    # stop_words
    [1, 2, 5],            # min_df
    [0.5, 1.0],          # max_df
))

# MultinomialNB: https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB
# ComplementNB: https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.ComplementNB.html#sklearn.naive_bayes.ComplementNB
def create_model(model_name: str, alpha: float):
    if model_name == 'MultinomialNB':
        return MultinomialNB(alpha=alpha)
    return ComplementNB(alpha=alpha)

configs_model = (
    [("MultinomialNB", alpha) for alpha in [0.05, 0.1, 0.5, 1.0]] +
    [("ComplementNB", alpha) for alpha in [0.05, 0.1, 0.5, 1.0]]
)

results = []
best_model = {'f1_score' : 0.0, 'y_pred_best': np.empty(0)}

for vectorizer_name, stop_words, min_df, max_df in configs_vect:

    vectorizer = create_vectorizer(vectorizer_name, stop_words, min_df, max_df)

    # We train TfidfVectorizer or CountVectorizer with different parameters
    Xtr = vectorizer.fit_transform(newsgroups_train.data)
    Xte = vectorizer.transform(newsgroups_test.data)

    for model_name, alpha in configs_model:
        model = create_model(model_name, alpha)
        model.fit(Xtr, y_train)
        y_hat = model.predict(Xte)
        model_f1_score = f1_score(y_test, y_hat, average="macro")

        if model_f1_score > best_model['f1_score']:
            best_model["y_pred_best"] = y_hat
            best_model["f1_score"] = model_f1_score

        results.append({
            "vectorizer": vectorizer_name, "stop_words": str(stop_words), "min_df": min_df, "max_df": max_df, "vocabulary": Xtr.shape[1], 
            "model": model_name, "alpha": alpha,
            "f1_macro": model_f1_score, "accuracy": accuracy_score(y_hat, y_test)
        })

results = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
print(f"{len(results)} configuraciones evaluadas")
results

192 configuraciones evaluadas


,vectorizer,stop_words,min_df,max_df,vocabulary,model,alpha,f1_macro,accuracy
0,tfidf,None,2,0.5,39414,ComplementNB,0.5,0.698490,0.715879
1,tfidf,None,2,1.0,39423,ComplementNB,0.5,0.698038,0.715481
2,tfidf,english,1,1.0,101322,ComplementNB,0.5,0.697805,0.716543
3,tfidf,english,1,0.5,101322,ComplementNB,0.5,0.697805,0.716543
4,tfidf,english,2,1.0,39115,ComplementNB,0.5,0.697363,0.714551
...,...,...,...,...,...,...,...,...,...
187,count,None,2,1.0,39423,MultinomialNB,1.0,0.588983,0.620287
188,tfidf,None,1,1.0,101631,MultinomialNB,1.0,0.585435,0.606213
189,count,None,1,1.0,101631,MultinomialNB,0.5,0.582580,0.615507
190,count,None,1,0.5,101622,MultinomialNB,1.0,0.547038,0.579926


Al evaluar todas las configuraciones vemos que el mejor f1-score macro se obtuvo con

- Vectorizer: `TfidfVectorizer(stop_words=None, min_df=2, max_df=0.5)`
- Modelo: `ComplementNB(alpha=0.5)`

Utilizando la mejor configuracion ahora lo comparamos con nuestro baseline

In [37]:
# Predicciones hechas por el mejor modelo (segun f1-score)
y_pred_best = best_model['y_pred_best']

print(f"F1-score macro baseline  (TfidfVectorizer() + MultinomialNB()) : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-score macro optimizado : {f1_score(y_test, y_pred_best, average='macro'):.4f}\n")

F1-score macro baseline  (TfidfVectorizer() + MultinomialNB()) : 0.5854
F1-score macro optimizado : 0.6985



Comparando el modelo baseline `MultinomialNB()` + `TfidfVectorizer()` (usando los valores por defecto del modelo y el vectorizador) con el modelo optimizado `TfidfVectorizer(stop_words=None, min_df=2, max_df=0.5)` + `ComplementNB(alpha=0.5)` vemos que nuestro f1-score macro sube de `0.5854` hasta `0.6985`. 

Esto representa una mejora significativa del 19.32% por lo que concluimos nuestro nuevo modelo es una mejora sobre el baseline original.

### Punto 4

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

Elegir las palabras **MANUALMENTE** para evitar la aparición de términos poco interpretables.

Al vectorizar creamos una matriz documento-termino en donde cada fila representa un documento vectorizado. Al transponer la matriz documento-termino obtenemos una matriz termino-documento en donde cada fila es ahora el vector de una palabra, y sus componentes son el peso TF-IDF de esa palabra en cada documento.

En este contexto; de la misma forma que usabamos la matriz documento-termino para encontrar documentos similares a un documento dado usando similitud coseno, usando la matriz termino-documento podemos obtener los terminos similares a un termino dado.

Para quedarnos con terminos interpretables usamos un vectorizador con `stop_words='english'` y `min_df=5` (descartamos palabras que aparecen en menos de 5 documentos). Elegimos una palabra por tematica del corpus.

In [38]:
def get_top_5_similar_words(similarity, idx2word):
    top_5_similar_index = np.argsort(similarity)[::-1][1:6]
    return {
        "top_5_similar_words": [idx2word[j] for j in top_5_similar_index],
        "similarity": [round(float(similarity[j]), 3) for j in top_5_similar_index]
    }

vect_palabras = TfidfVectorizer(stop_words="english", min_df=5)

# X_train es mi matriz documento-termino
# Cada fila tiene un documento y cada columna es un termino (token)
X_train = vect_palabras.fit_transform(newsgroups_train.data)

# Al trasponer la matriz documento-termino me queda la matriz termino-documento
# Cada fila tiene un termino (token) y cada columna es un documento (interpretamos como peso de esa palabra en los doc)
term_doc = X_train.T.tocsr()


idx2word = {index: word for word, index in vect_palabras.vocabulary_.items()}

# Elegimos manualmente una palabra por tematica
selected_words = ["god", "car", "space", "gun", "windows"]

data = [
    {
        "word": word,
        **get_top_5_similar_words(
            cosine_similarity(term_doc[vect_palabras.vocabulary_[word]], term_doc)[0], 
            idx2word
        )
    } for word in selected_words 
]

print(f"Matriz termino-documento: {term_doc.shape}")
pd.DataFrame(data)

Matriz termino-documento: (17797, 11314)


,word,top_5_similar_words,similarity
0,god,"[jesus, bible, christ, faith, existence]","[0.277, 0.268, 0.267, 0.255, 0.249]"
1,car,"[cars, dealer, civic, loan, owner]","[0.192, 0.177, 0.163, 0.156, 0.148]"
2,space,"[nasa, shuttle, exploration, aeronautics, cfa]","[0.318, 0.278, 0.233, 0.222, 0.216]"
3,gun,"[guns, handgun, crime, firearms, homicides]","[0.375, 0.245, 0.243, 0.238, 0.231]"
4,windows,"[dos, ms, microsoft, nt, file]","[0.308, 0.225, 0.207, 0.197, 0.193]"


Observamos que aunque para ningun termino se haya obtenido una similaridad mayor a 0.4 vemos que las palabras similares bastante buenos.

Podriamos intentar buscar hiperparametros que mejoren al vectorizador, sin embargo el mayor limitante para este modelo es que el corpus utilizado para entrenar sea suficientemente representativo. Para obtener mejores terminos, se recomienda empezar por agregar mas documentos limpios